# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

#### Load extra jars

In [1]:
%extra_jars s3://orders-realtime-bucket/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar,s3://orders-realtime-bucket/jars/kafka-clients-3.3.1.jar


UsageError: Line magic function `%extra_jars` not found.


####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 2bbc6ab7-d13b-4c61-8c68-98bbb9dff32e
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--extra-jars s3://prepzee-streaming/jars/spark-sql-kafka-0-10_2.12-3.3.0.jar,s3://prepzee-streaming/jars/kafka-clients-3.3.1.jar
Waiting for session 2bbc6ab7-d13b-4c61-8c68-98bbb9dff32e to get into ready status...
Session 2bbc6ab7-d13b-4c61-8c68-98bbb9dff32e has been created.



#### Test the Kafka Connection


In [ ]:

# -------------------------------------------------
# 4. TEST KAFKA READ (batch mode for quick validation)
# -------------------------------------------------
# Use batch mode first to verify the connector loads correctly.
# Switch to readStream once this works.
test_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "pkc-n98pk.us-west-2.aws.confluent.cloud:9092") \
    .option("subscribe", "orders_topic") \
    .option("startingOffsets", "earliest") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config",
            f'org.apache.kafka.common.security.plain.PlainLoginModule required username="GIITD77LNNPTLUPY" password="cfltGr8vg7/QUyxTssw3J2azvM7LLVn7DuZ/lVr+XscdG5uY/X9mEp3g6v6fG80g";') \
    .load()

test_df.printSchema()
print(">>> Kafka connector loaded successfully.")
test_df.show(5, truncate=False)

#### Stream Kafka data into Spark DataFrame


In [ ]:
from pyspark.sql.functions import col, from_json, to_date, current_timestamp, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# --- Define your JSON schema ---
order_schema = StructType([
    StructField("order_id",      StringType(),  True),
    StructField("customer_id",   StringType(),  True),
    StructField("product",       StringType(),  True),
    StructField("quantity",      IntegerType(), True),
    StructField("price",         DoubleType(),  True),
    StructField("order_timestamp", TimestampType(), True)
])

# --- Read from Kafka ---
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "pkc-n98pk.us-west-2.aws.confluent.cloud:9092") \
    .option("subscribe", "orders_topic") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config",
            f'org.apache.kafka.common.security.plain.PlainLoginModule required username="GIITD77LNNPTLUPY" password="cfltGr8vg7/QUyxTssw3J2azvM7LLVn7DuZ/lVr+XscdG5uY/X9mEp3g6v6fG80g";') \
    .load()

# --- Parse & clean ---
df_parsed = df_raw.select(
    from_json(col("value").cast("string"), order_schema).alias("data")
).select("data.*")

# Add helper columns for partitioning / auditing
df_clean = df_parsed \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("order_date", to_date(col("order_timestamp")))

#### Write to S3

In [ ]:
S3_OUTPUT     = "s3://orders-realtime-bucket/notebook-orders/" 
CHECKPOINT_S3 = "s3://orders-realtime-bucket/checkpoints/notebook-orders/"

query_s3 = df_clean.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", S3_OUTPUT) \
    .option("checkpointLocation", CHECKPOINT_S3) \
    .partitionBy("order_date") \
    .trigger(processingTime="30 seconds") \
    .start()

query_s3.awaitTermination()

Execution Interrupted. Attempting to cancel the statement (statement_id=5)
